# 02 - Train Piper TTS (Vietnamese Male Voice)

Notebook này chạy mỗi session Colab để fine-tune Piper TTS từ vais1000-medium.

**Trước khi chạy**:
1. Đã chạy `00_clone_repo.ipynb` (repo có trên Drive).
2. Đã chạy `01_bootstrap_env.ipynb` ít nhất 1 lần (venv tồn tại hoặc snapshot có trên Drive).
3. Dataset đã preprocess xong (xem CLAUDE.md Phase 3-4) — có file `config.json` + `dataset.jsonl` trong `training_dir/`.

**Skeleton** — sẽ fill in chi tiết sau khi dataset sẵn sàng.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import subprocess

BASE_DIR = '/content/drive/MyDrive/VKU/Graduation_project'
REPO_DIR = f'{BASE_DIR}/piper_vi_vais1000_finetuning'
DRIVE_ROOT = f'{BASE_DIR}/piper_vi_vais1000_finetuning_colab'
VENV_DIR = '/content/venvs/piper_finetune'
BOOTSTRAP = f'{REPO_DIR}/colab/scripts/bootstrap_env.sh'

# Auto-bootstrap nếu runtime mới (venv chưa có)
if not os.path.isfile(f'{VENV_DIR}/bin/python'):
    print('[INFO] Venv chua co. Chay bootstrap (restore tu snapshot neu co)...')
    env = os.environ.copy()
    env.update({
        'BASE_ROOT': BASE_DIR,
        'REPO_DIR': REPO_DIR,
        'DRIVE_ROOT': DRIVE_ROOT,
        'VENV_DIR': VENV_DIR,
        'ENABLE_VENV_SNAPSHOT': '1',
        'FORCE_REINSTALL': '0',
    })
    subprocess.run(['bash', BOOTSTRAP], check=True, env=env)

assert os.path.isfile(f'{VENV_DIR}/bin/python'), 'Bootstrap failed'
print('Repo dir   :', REPO_DIR)
print('Drive root :', DRIVE_ROOT)
print('Venv dir   :', VENV_DIR)

## Bước 1: Tải base checkpoint vais1000-medium

Chỉ chạy LẦN ĐẦU. Sau đó file ở `<DRIVE_ROOT>/checkpoints/` được giữ lại.

In [ ]:
%%bash
set -euo pipefail
DRIVE_ROOT=/content/drive/MyDrive/VKU/Graduation_project/piper_vi_vais1000_finetuning_colab
VENV=/content/venvs/piper_finetune
CKPT_DIR="$DRIVE_ROOT/checkpoints/vais1000_medium"
mkdir -p "$CKPT_DIR"

if [ -f "$CKPT_DIR/epoch.ckpt" ] || ls "$CKPT_DIR"/*.ckpt 2>/dev/null; then
    echo '[SKIP] Checkpoint da co'
else
    "$VENV/bin/python" <<'PY'
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id="rhasspy/piper-checkpoints",
    repo_type="dataset",
    allow_patterns=["vi/vi_VN/vais1000/medium/*"],
    local_dir="/content/drive/MyDrive/VKU/Graduation_project/piper_vi_vais1000_finetuning_colab/checkpoints",
    local_dir_use_symlinks=False,
)
PY
fi
echo
ls -la "$DRIVE_ROOT/checkpoints/vi/vi_VN/vais1000/medium/"

## Bước 2: Preprocess dataset

**Yêu cầu**: thư mục `<DRIVE_ROOT>/dataset/` có sẵn:
- `wav/` — file `00001.wav`, `00002.wav`, ... (sample rate ≥ 22050).
- `metadata.csv` — format `id|text` (UTF-8, no header).

Output: `<DRIVE_ROOT>/training_dir/` với `config.json`, `dataset.jsonl`, các file `.pt`.

In [ ]:
%%bash
set -euo pipefail
DRIVE_ROOT=/content/drive/MyDrive/VKU/Graduation_project/piper_vi_vais1000_finetuning_colab
VENV=/content/venvs/piper_finetune

"$VENV/bin/python" -m piper_train.preprocess \
    --language vi \
    --input-dir "$DRIVE_ROOT/dataset" \
    --output-dir "$DRIVE_ROOT/training_dir" \
    --dataset-format ljspeech \
    --single-speaker \
    --sample-rate 22050

**Quan trọng**: ghi đè `config.json` của training_dir bằng config của vais1000-medium để match phoneme_id_map.

In [ ]:
%%bash
set -euo pipefail
DRIVE_ROOT=/content/drive/MyDrive/VKU/Graduation_project/piper_vi_vais1000_finetuning_colab
CKPT_DIR="$DRIVE_ROOT/checkpoints/vi/vi_VN/vais1000/medium"

# Backup config gốc của preprocess (chứa speaker info)
cp "$DRIVE_ROOT/training_dir/config.json" "$DRIVE_ROOT/training_dir/config.preprocess.json"

# Copy config vais1000 (phoneme_id_map match base)
cp "$CKPT_DIR/config.json" "$DRIVE_ROOT/training_dir/config.json"

echo '[OK] Da thay config.json bang config vais1000'
head -50 "$DRIVE_ROOT/training_dir/config.json"

## Bước 3: Train (fine-tune cross-gender)

In [ ]:
%%bash
set -euo pipefail
DRIVE_ROOT=/content/drive/MyDrive/VKU/Graduation_project/piper_vi_vais1000_finetuning_colab
VENV=/content/venvs/piper_finetune
CKPT_DIR="$DRIVE_ROOT/checkpoints/vi/vi_VN/vais1000/medium"

# Tu dong tim file .ckpt moi nhat (epoch cao nhat)
BASE_CKPT=$(ls -t "$CKPT_DIR"/*.ckpt 2>/dev/null | head -1)
if [ -z "$BASE_CKPT" ]; then
    echo '[ERROR] Khong tim thay vais1000 checkpoint trong $CKPT_DIR'
    exit 1
fi
echo "Base checkpoint: $BASE_CKPT"

# Neu da co training run dang do, resume tu day. Neu chua, dung BASE_CKPT.
RESUME_CKPT="$BASE_CKPT"
LATEST=$(ls -t "$DRIVE_ROOT/training_dir/lightning_logs/version_*/checkpoints/*.ckpt" 2>/dev/null | head -1)
if [ -n "$LATEST" ]; then
    echo "[INFO] Found in-progress checkpoint: $LATEST. Resuming."
    RESUME_CKPT="$LATEST"
fi

"$VENV/bin/python" -m piper_train \
    --dataset-dir "$DRIVE_ROOT/training_dir" \
    --accelerator gpu \
    --devices 1 \
    --batch-size 16 \
    --validation-split 0.0 \
    --num-test-examples 0 \
    --max_epochs 3000 \
    --resume_from_checkpoint "$RESUME_CKPT" \
    --checkpoint-epochs 1 \
    --precision 32 \
    --max-phoneme-ids 400 \
    --quality medium 2>&1 | tee -a "$DRIVE_ROOT/logs/train.log"

## Bước 4: TensorBoard monitoring

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/VKU/Graduation_project/piper_vi_vais1000_finetuning_colab/training_dir/lightning_logs